# SAP Production Delay Prediction - Quick Start

This notebook demonstrates the basic workflow:
1. Load sample data
2. Feature engineering
3. Train XGBoost model
4. Make predictions

In [ ]:
# Import libraries
import sys
sys.path.append('..')

from src.data_collection.csv_loader import CSVLoader
from src.data_processing.feature_engineer import FeatureEngineer
from src.models.xgboost_model import ProductionDelayModel

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_style('whitegrid')

## 1. Load Data

In [ ]:
# Load sample data
loader = CSVLoader(data_dir='../data/sample')
df = loader.load_production_orders()

print(f"Loaded {len(df)} orders")
df.head()

In [ ]:
# Data info
df.info()

## 2. Feature Engineering

In [ ]:
# Apply feature engineering
engineer = FeatureEngineer(lookback_days=30)
df_features = engineer.transform(df)

print(f"Total features: {len(df_features.columns)}")
df_features.head()

In [ ]:
# Check target distribution
if 'is_delayed' in df_features.columns:
    print("\nTarget Distribution:")
    print(df_features['is_delayed'].value_counts())
    
    # Visualize
    plt.figure(figsize=(8, 5))
    df_features['is_delayed'].value_counts().plot(kind='bar')
    plt.title('Order Delay Distribution')
    plt.xlabel('Delayed (0=No, 1=Yes)')
    plt.ylabel('Count')
    plt.xticks(rotation=0)
    plt.show()

## 3. Train Model

In [ ]:
# Prepare training data (only orders with actual_finish)
df_train = df_features[df_features['actual_finish'].notna()].copy()

feature_cols = engineer.get_feature_names()
df_train = df_train.dropna(subset=feature_cols + ['is_delayed'])

X = df_train[feature_cols].values
y = df_train['is_delayed'].values

print(f"Training samples: {len(X)}")
print(f"Features: {X.shape[1]}")

In [ ]:
# Split data
model = ProductionDelayModel()
X_train, X_val, y_train, y_val = model.split_data(X, y, test_size=0.2)

print(f"Train: {len(X_train)}, Validation: {len(X_val)}")

In [ ]:
# Train model
metrics = model.train(X_train, y_train, X_val, y_val, early_stopping_rounds=10)

print("\nValidation Metrics:")
for metric, value in metrics.items():
    if metric != 'confusion_matrix':
        print(f"{metric:15s}: {value:.4f}")

## 4. Feature Importance

In [ ]:
# Get feature importance
importance = model.get_feature_importance()
importance_df = pd.DataFrame([
    {'feature': k, 'importance': v} 
    for k, v in importance.items()
]).sort_values('importance', ascending=False)

# Visualize
plt.figure(figsize=(10, 6))
plt.barh(importance_df['feature'], importance_df['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 5. Make Predictions

In [ ]:
# Predict on validation set
y_pred = model.predict(X_val)
y_proba = model.predict_proba(X_val)[:, 1]

# Create results dataframe
results_df = pd.DataFrame({
    'actual': y_val,
    'predicted': y_pred,
    'probability': y_proba
})

results_df['risk_level'] = pd.cut(
    results_df['probability'],
    bins=[0, 0.3, 0.7, 1.0],
    labels=['Low', 'Medium', 'High']
)

results_df.head(10)

In [ ]:
# Confusion matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_val, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['On-time', 'Delayed'])

fig, ax = plt.subplots(figsize=(8, 6))
disp.plot(ax=ax, cmap='Blues')
plt.title('Confusion Matrix')
plt.show()

## 6. Save Model

In [ ]:
# Save model
model.save('../models/xgb_model_demo.json')
print("Model saved successfully!")